# Hierarchical Forecasting

**Hierarquia:** Total → Regiões (R1, R2) → Folhas (A1, A2, B1, B2)

**Método:** seasonal naive nas folhas + reconciliação *bottom-up* vs. forecast direto do Total.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

periods = 100
t = np.arange(periods)

def series(level, slope, amp, phase, noise=0.3):
    seasonal = amp * np.sin(2*np.pi*t/12 + phase)
    trend = level + slope*t
    return trend + seasonal + rng.normal(0, noise, periods)

leaves = {
    'A1': series(100, 0.5, 8, 0.0),
    'A2': series(80, 0.4, 6, 1.2),
    'B1': series(120, 0.6, 10, 2.3),
    'B2': series(90, 0.3, 5, 0.8),
}
df = pd.DataFrame(leaves)
df['R1'] = df['A1'] + df['A2']
df['R2'] = df['B1'] + df['B2']
df['Total'] = df['R1'] + df['R2']

print(df.head(3).round(2).to_string())

       A1     A2      B1     B2      R1      R2   Total
0   99.76  85.01  127.51  93.60  184.77  221.11  405.87
1  104.57  86.32  123.85  94.84  190.89  218.69  409.58
2  107.36  85.76  119.17  95.74  193.12  214.90  408.03


In [2]:
season = 12
train_len = 80
h = 20

def seasonal_naive(y, h, season=season):
    n = len(y)
    preds = []
    for i in range(h):
        k = i % season
        block = i // season + 1
        idx = n - block*season + k
        preds.append(y[idx])
    return np.array(preds)

def bottom_up(df, train_len, h):
    fc = {}
    for leaf in ['A1', 'A2', 'B1', 'B2']:
        fc[leaf] = seasonal_naive(df[leaf].iloc[:train_len].values, h)
    fc['R1'] = fc['A1'] + fc['A2']
    fc['R2'] = fc['B1'] + fc['B2']
    fc['Total'] = fc['R1'] + fc['R2']
    return fc

fc_bu = bottom_up(df, train_len, h)

In [3]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

y_total_true = df['Total'].iloc[train_len:train_len+h].values
y_total_train = df['Total'].iloc[:train_len].values

fc_total_bu = fc_bu['Total']
fc_total_direct = seasonal_naive(y_total_train, h)

mape_bu = mape(y_total_true, fc_total_bu)
mape_direct = mape(y_total_true, fc_total_direct)

print('=' * 55)
print('HIERARCHICAL FORECAST — RESULTADOS')
print('=' * 55)
print(f'MAPE bottom-up (Total): {mape_bu:.2f}%')
print(f'MAPE direct    (Total): {mape_direct:.2f}%')
print(f'Melhora (pp): {mape_direct - mape_bu:+.2f}')

ok1 = np.allclose(fc_bu['R1'], fc_bu['A1'] + fc_bu['A2'])
ok2 = np.allclose(fc_bu['Total'], fc_bu['R1'] + fc_bu['R2'])
print()
print('Reconciliação R1 == A1+A2 :', ok1)
print('Reconciliação Total == R1+R2 :', ok2)

comp = pd.DataFrame({
    'real': y_total_true,
    'bottom_up': fc_total_bu,
    'direct': fc_total_direct,
})
print()
print('Primeiras 6 linhas da previsão do Total:')
print(comp.head(6).round(2).to_string())

HIERARCHICAL FORECAST — RESULTADOS
MAPE bottom-up (Total): 6.94%
MAPE direct    (Total): 6.94%
Melhora (pp): +0.00

Reconciliação R1 == A1+A2 : True
Reconciliação Total == R1+R2 : True

Primeiras 6 linhas da previsão do Total:
     real  bottom_up  direct
0  521.28     497.02  497.02
1  528.49     507.45  507.45
2  539.60     517.80  517.80
3  549.80     529.00  529.00
4  558.91     536.57  536.57
5  561.25     540.21  540.21
